# Tema 2: Manejo de conectores

## Creacción de conexiones

### SQLite

In [5]:
import sqlite3
from contextlib import closing
from datetime import date

with closing(sqlite3.connect('usuarios.db')) as con:
    with con:
        con.execute('CREATE TABLE IF NOT EXISTS usuarios (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, nombre TEXT, contrasenia TEXT)')


# con.close()

In [9]:
with closing(sqlite3.connect('usuarios.db')) as con:
    with con:
        con.execute('DROP TABLE IF EXISTS usuarios2')
        con.execute('DROP TABLE IF EXISTS test2')

In [27]:
from dataclasses import dataclass

@dataclass()
class Usuario:
    nombre : str
    email : str
    contrasenia : str

    @classmethod
    def desde_dict(cls, datos):
         # return cls(**datos) # En el caso de que datos = {'nombre' : ... , 'email' : ... , 'contrasenia' : ...}
        return cls(nombre = datos['nombre'],
                    email = datos['email'],
                    contrasenia = datos['contrasenia'])

usuario = Usuario.desde_dict({'nombre' : 'Jaime', 'email' : 'jaime.vilber@gmail.com', 'contrasenia' : 1234, 'test' : True})

with closing(sqlite3.connect('usuarios.db')) as con:
    with con:
        query = 'INSERT INTO usuarios(email, nombre, contrasenia) VALUES ( ?, ?, ?)'
        print(query)
        con.execute(query, (usuario.email, usuario.nombre, usuario.contrasenia))


INSERT INTO usuarios(email, nombre, contrasenia) VALUES ( ?, ?, ?)


In [19]:
with closing(sqlite3.connect('usuarios.db')) as con:
    with con:
        query = 'DELETE FROM usuarios WHERE id == 1'
        print(query)
        con.execute(query)

DELETE FROM usuarios WHERE id == 1


In [24]:
with closing(sqlite3.connect('usuarios.db')) as con:
    with con:
        query = 'SELECT * FROM sqlite_sequence'
        print(query)
        res = con.execute(query)
        for l in res:
            print(l)

SELECT * FROM sqlite_sequence
('usuarios', 1)


### MySQL

In [1]:
import mysql.connector

In [7]:
with mysql.connector.connect(host="localhost", port=3306, user='root', password='1234') as con:
    with con.cursor() as cursor:
        cursor.execute('CREATE DATABASE IF NOT EXISTS AAD')
        cursor.execute('CREATE TABLE IF NOT EXISTS AAD.usuarios (id INT, email VARCHAR(50), nombre VARCHAR(50), contrasenia VARCHAR(50))')

In [10]:
# Uso .env
import os
import dotenv

dotenv.load_dotenv()
host = os.getenv('HOST')
puerto = os.getenv('PORT')
usuario = os.getenv('USER')
contrasenia = os.getenv('PASS')

with mysql.connector.connect(host=host,
                             port=puerto,
                             user=usuario,
                             password=contrasenia) as con:
    pass


localhost


In [8]:
import mysql.connector

config = {"host" : "localhost",
          "port": 3306,
          "user": "root",
          "password" : "1234"
          }
with mysql.connector.connect(**config) as conn:
    with conn.cursor() as cur:
        cur.execute("SHOW DATABASES")
        for schema in cur:
            print(schema)
        cur.execute('CREATE SCHEMA test')

('information_schema',)
('mysql',)
('performance_schema',)
('sys',)


### ODBC

In [14]:
import pyodbc

# print(pyodbc.dataSources())

with pyodbc.connect('DSN=Test;') as con:
    with con.cursor() as cursor:
        res = cursor.execute('SHOW TABLES')
        for row in res:
            print(row)

('usuarios',)


In [15]:
pyodbc.drivers()

['SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'MySQL ODBC 26.10 ANSI Driver',
 'MySQL ODBC 26.10 Unicode Driver']

In [17]:
cadena_conexion = 'DRIVER={MySQL ODBC 26.10 Unicode Driver};SERVER=127.0.0.1;PORT=3306;DATABASE=AAD;UID=root;PWD=1234;'

with pyodbc.connect(cadena_conexion) as con:
    with con.cursor() as cursor:
        res = cursor.execute('SHOW TABLES')
        for row in res:
            print(row)

('usuarios',)


## Insertar

### Sqlite

In [2]:
from contextlib import closing
import sqlite3

with closing(sqlite3.connect('DAM.db')) as con:
    cur = con.execute('CREATE TABLE IF NOT EXISTS alumnos ( '
                      ' nombre TEXT,'
                      ' apellidos TEXT,'
                      ' dni TEXT PRIMARY KEY,'
                      ' curso_matriculado INTEGER,'
                      ' es_repetidor INTEGER,'
                      ' fecha_nacimiento TEXT )')

In [66]:
from dataclasses import dataclass, asdict
from contextlib import closing
import datetime
import sqlite3
from sqlite3 import IntegrityError

@dataclass()
class Alumno:
    nombre: str
    apellidos: str
    dni: str
    curso_matriculado: int
    es_repetidor: bool
    fecha_nacimiento: date

class AlumnoDAO:
    def __init__(self, ruta_bd):
        self.ruta_bd = ruta_bd
        with closing(sqlite3.connect(self.ruta_bd)) as con:
            con.execute('CREATE TABLE IF NOT EXISTS alumnos ( '
                      ' nombre TEXT,'
                      ' apellidos TEXT,'
                      ' dni TEXT PRIMARY KEY,'
                      ' curso_matriculado INTEGER,'
                      ' es_repetidor INTEGER,'
                      ' fecha_nacimiento TEXT )')

    def crear(self, alumno: Alumno) -> bool:
        with closing(sqlite3.connect(self.ruta_bd)) as con:
            with con:
                #query = 'INSERT INTO alumnos VALUES (?, ?, ?, ?, ?, ?)'
                #con.execute(query, (alumno.nombre, alumno.apellidos, alumno.dni, alumno.curso_matriculado, alumno.es_repetidor, str(alumno.fecha_nacimiento) ) )

                query = 'INSERT INTO alumnos VALUES (:nombre, :apellidos, :dni, :curso_matriculado, :es_repetidor, :fecha_nacimiento)'

                alumno = asdict(alumno)
                alumno['fecha_nacimiento'] = str(alumno['fecha_nacimiento'])

                try:
                    cur = con.execute(query, alumno)
                except Exception:
                    resultado = False
                else:
                    resultado = True

        return resultado

    def crear_varios(self, alumnos: list[Alumno]) -> int:
        with closing(sqlite3.connect(self.ruta_bd)) as con:
            with con:
                #query = 'INSERT INTO alumnos VALUES (?, ?, ?, ?, ?, ?)'
                #con.execute(query, (alumno.nombre, alumno.apellidos, alumno.dni, alumno.curso_matriculado, alumno.es_repetidor, str(alumno.fecha_nacimiento) ) )

                query = 'INSERT INTO alumnos VALUES (:nombre, :apellidos, :dni, :curso_matriculado, :es_repetidor, :fecha_nacimiento)'

                alumnos_modificados: list[dict] = []
                for alumno in alumnos:
                    alumno_dict = asdict(alumno)
                    alumno_dict['fecha_nacimiento'] = str(alumno_dict['fecha_nacimiento'])
                    alumnos_modificados.append(alumno_dict)

                print(alumnos_modificados)
                cur = con.executemany(query, alumnos_modificados )

        return cur.rowcount

    def borrar_todos(self):
        with closing(sqlite3.connect(self.ruta_bd)) as con:
            with con:
                query = 'DELETE FROM alumnos'
                con.execute(query)

    def obtener_todos(self):
        with closing(sqlite3.connect(self.ruta_bd)) as con:
            cur = con.execute('SELECT * FROM alumnos')
            alumno = cur.fetchone()
            print(alumno)
            alumno = cur.fetchone()
            print(alumno)
            alumno = cur.fetchone()
            print(alumno)
            # for alumno in cur:
            #     print(alumno)


alumno = Alumno('Pepe', 'Viyuela', '123456789K', 2, True, datetime.date(1799, 1, 1) )
alumno2 = Alumno('Juan', 'Cruz', '987654321W', 1, False, datetime.date(1896, 1, 1) )
alumnoDAO = AlumnoDAO('DAM.db')
alumnoDAO.borrar_todos()
# alumnoDAO.crear(alumno)
print(alumnoDAO.crear_varios([alumno, alumno2]))
alumnoDAO.obtener_todos()

[{'nombre': 'Pepe', 'apellidos': 'Viyuela', 'dni': '123456789K', 'curso_matriculado': 2, 'es_repetidor': True, 'fecha_nacimiento': '1799-01-01'}, {'nombre': 'Juan', 'apellidos': 'Cruz', 'dni': '987654321W', 'curso_matriculado': 1, 'es_repetidor': False, 'fecha_nacimiento': '1896-01-01'}]
2
('Pepe', 'Viyuela', '123456789K', 2, 1, '1799-01-01')
('Juan', 'Cruz', '987654321W', 1, 0, '1896-01-01')
None


In [39]:
with closing(sqlite3.connect('DAM.db')) as con:
    with con:
        con.execute('CREATE TABLE IF NOT EXISTS test ( numero INTEGER )')
        con.execute('INSERT INTO test (numero) VALUES ( :n )', { 'n' : 2}  )
        cur = con.execute('SELECT * FROM test')
        for linea in cur:
            print(linea)


(1,)
(2,)


In [58]:
with closing(sqlite3.connect('DAM.db')) as con:
    with con:
        con.execute('CREATE TABLE IF NOT EXISTS test ( numero INTEGER )')
        con.executemany('INSERT INTO test (numero) VALUES ( ? )', ( (1,), (2,), (3,) ) )
        cur = con.execute('SELECT * FROM test')
        for linea in cur:
            print(linea)


(1,)
(2,)
(3,)


In [40]:
alumno = Alumno('Pepe', 'Viyuela', '123456789J', 2, True, datetime.date(1956, 8, 26) )
print(asdict(alumno))

{'nombre': 'Pepe', 'apellidos': 'Viyuela', 'dni': '123456789J', 'curso_matriculado': 2, 'es_repetidor': True, 'fecha_nacimiento': datetime.date(1956, 8, 26)}


### Mysql

In [54]:
# Uso .env
import os
import dotenv
import mysql.connector

dotenv.load_dotenv()
host = os.getenv('HOST')
puerto = os.getenv('PORT')
usuario = os.getenv('USER')
contrasenia = os.getenv('PASS')

with mysql.connector.connect(host=host,
                             port=puerto,
                             user=usuario,
                             password=contrasenia,
                             autocommit = True) as con:
    with con.cursor() as cur:
        cur.execute('CREATE TABLE IF NOT EXISTS DAM.test ( numero INTEGER )')
        # query = 'INSERT INTO DAM.test (numero) VALUES ( %s )'
        # cur.execute(query, (3, ) )
        query = 'INSERT INTO DAM.test (numero) VALUES ( %(n)s )'
        cur.execute(query, {'n': 3})
        con.commit()

